# Modul 2 — MASTER Kelompok: Server Laporan & Monitor Cluster
Satu kelompok = **5 mahasiswa**: **1 PC Master** + **4 PC Worker**.
Notebook ini dijalankan **hanya di PC Master kelompok**. Kelima mahasiswa (termasuk pemegang Master) boleh menjalankan `Modul2-Cluster.ipynb` sebagai Driver.

```
            PC Master (Spark Master :7077, UI :8080, monitor :8090)
          /          |            |           \
   PC Worker 1   PC Worker 2   PC Worker 3   PC Worker 4
   (Worker +     (Worker +     (Worker +     (Worker +
    Driver)       Driver)       Driver)       Driver)
```

| Sel | Fungsi |
|---|---|
| **Sel 1** (sekali) | Konfigurasi + server laporan task di port **8090** (tetap hidup di background) |
| **Sel 2** | Monitor real-time. Hentikan dengan tombol **Interrupt (■)**, lalu bisa dijalankan ulang kapan saja |
| **Sel 3** (opsional) | Reset laporan / matikan server |

Panel monitor:
- **[A] Cluster Manager & Worker**: status Master, worker ALIVE, core terpakai/total (dari `:8080/json/`)
- **[B] Aplikasi**: 1 mahasiswa = 1 Driver = 1 Application, jumlah executor, job, dan task (dari Spark UI Driver `:4040`)
- **[C] Detail satu mahasiswa** (isi `FOCUS`): Job → Stage (termasuk SKIPPED dan shuffle) → Executor
- **[D] Laporan task**: dikirim langsung oleh executor (fungsi `pantau()` di notebook mahasiswa), menunjukkan worker mana yang mengerjakan

## Langkah 0: Start Master & Worker (terminal / CMD)
Ganti `10.73.3.90` dengan **IP PC Master kelompok Anda** (cek dengan `ipconfig`). Semua anggota kelompok memakai IP yang sama.

**PC Master kelompok** (1 orang):
```bat
%SPARK_HOME%\bin\spark-class org.apache.spark.deploy.master.Master --host 10.73.3.90 --port 7077 --webui-port 8080
```
**4 PC Worker** (masing-masing 1 terminal, biarkan tetap terbuka):
```bat
%SPARK_HOME%\bin\spark-class org.apache.spark.deploy.worker.Worker spark://10.73.3.90:7077 --cores 4 --memory 4g
```
`--cores` = jumlah core yang disumbangkan (≤ core fisik PC). Dengan 4 worker × 4 core = **16 core** untuk kelompok.
Kalau PC hanya punya 2 core, pakai `--cores 2 --memory 2g` (total 8 core).

Cek di browser `http://10.73.3.90:8080`: **Alive Workers = 4**.
Firewall PC Master: buka port **7077, 8080, 8090**. Firewall PC Worker: izinkan Java/Python (port 4040 dan port executor).

In [1]:
# ===== SEL 1: Konfigurasi + server laporan (jalankan sekali) =====
import json
import time
import threading

from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import urlopen

MASTER = "26.66.202.158"                        # IP PC Master KELOMPOK, sesuaikan
MASTER_JSON = f"http://{MASTER}:8080/json/"  # Web UI Master (port 8080, BUKAN 7077)
PORT = 8090                                  # server laporan task
REFRESH = 2                                  # detik

# Pertahankan data jika sel dijalankan ulang
try:
    laporan, drivers, lock
except NameError:
    laporan = {}   # (driver, run, task, worker) -> laporan task dari executor
    drivers = {}   # app_id -> registrasi Driver mahasiswa (berisi alamat Spark UI)
    lock = threading.Lock()


def _balas(handler, code, teks):
    body = teks.encode("utf-8")
    handler.send_response(code)
    handler.send_header("Content-Type", "text/plain; charset=utf-8")
    handler.send_header("Content-Length", str(len(body)))
    handler.end_headers()
    handler.wfile.write(body)


class TaskHandler(BaseHTTPRequestHandler):

    def do_POST(self):
        try:
            size = int(self.headers.get("Content-Length", 0))
            if size <= 0:
                return self.send_error(400, "Request body kosong")

            data = json.loads(self.rfile.read(size))
            data["waktu"] = time.time()

            if self.path.rstrip("/") == "/driver":
                # Registrasi Driver: notebook mahasiswa mengirim alamat Spark UI (IP:4040)
                if not all(k in data for k in ("app_id", "host", "ui")):
                    return self.send_error(400, "Field driver tidak lengkap")
                with lock:
                    drivers[data["app_id"]] = data
            else:
                # Laporan task dari Executor
                required = ("run", "task", "worker", "app", "status")
                if not all(k in data for k in required):
                    return self.send_error(400, "Field laporan tidak lengkap")
                key = (data.get("driver", "?"), data["run"], data["task"], data["worker"])
                with lock:
                    laporan[key] = data

            _balas(self, 200, "OK")

        except (ValueError, KeyError, TypeError):
            self.send_error(400, "Format laporan tidak valid")
        except Exception:
            self.send_error(500, "Terjadi kesalahan internal")

    def do_GET(self):
        # Tes koneksi dari PC mahasiswa / browser
        with lock:
            n, d = len(laporan), len(drivers)
        _balas(self, 200, f"Monitor aktif. Driver terdaftar: {d} | Laporan task: {n}\n")

    def log_message(self, format, *args):
        pass


# Cegah "Address already in use" saat sel dijalankan ulang
try:
    server.shutdown()
    server.server_close()
except NameError:
    pass

ThreadingHTTPServer.allow_reuse_address = True
ThreadingHTTPServer.daemon_threads = True
server = ThreadingHTTPServer(("0.0.0.0", PORT), TaskHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()

print(f"Server laporan berjalan pada http://{MASTER}:{PORT}")
print(f"Master UI                  : http://{MASTER}:8080")

Server laporan berjalan pada http://26.66.202.158:8090
Master UI                  : http://26.66.202.158:8080


In [ ]:
# ===== SEL 2: Monitor real-time (hentikan dengan tombol Interrupt ■) =====
from IPython.display import clear_output

FOCUS = ""                 # sebagian hostname mahasiswa (mis. "PC-07") -> panel [C] + filter [D]
SHOW_IDLE_WORKERS = True   # kelompok hanya 4 worker -> tampilkan semua, termasuk yang menganggur
MAX_BARIS = 15             # batas baris per tabel agar layar tidak kepanjangan
W = 90


def get_json(url, timeout=2):
    try:
        with urlopen(url, timeout=timeout) as response:
            return json.load(response)
    except Exception:
        return None


def bar(used, total, width=10):
    if total <= 0:
        return "." * width
    filled = round(width * used / total)
    return "#" * filled + "." * (width - filled)


def ukuran(b):
    b = float(b or 0)
    for satuan in ("B", "KB", "MB", "GB"):
        if b < 1024:
            return f"{b:.0f} {satuan}" if satuan == "B" else f"{b:.1f} {satuan}"
        b /= 1024
    return f"{b:.1f} TB"


def ui_aplikasi(app):
    """Alamat Spark UI Driver: utamakan registrasi (pakai IP), fallback ke JSON Master."""
    with lock:
        reg = drivers.get(app.get("id"))
    if reg:
        return reg["ui"]
    ui = app.get("webuiaddress") or ""
    if not ui:
        return None
    return ui if ui.startswith("http") else "http://" + ui


def baca_aplikasi(app):
    base = ui_aplikasi(app)
    if base is None:
        return None
    api = f"{base}/api/v1/applications/{app['id']}"
    jobs = get_json(api + "/jobs")
    if jobs is None:
        return None   # Spark UI Driver (4040) tidak terjangkau
    stages = get_json(api + "/stages") or []
    execs = [e for e in (get_json(api + "/executors") or []) if e.get("id") != "driver"]
    return {"jobs": jobs, "stages": stages, "execs": execs}


pool = ThreadPoolExecutor(max_workers=16)   # ambil data semua Driver secara paralel


def tampilkan():
    print("=" * W)
    print(" MONITOR SPARK CLUSTER KELOMPOK (1 Master + 4 Worker)")
    print("=" * W)

    # ---------------- [A] CLUSTER MANAGER + WORKER ----------------
    data = get_json(MASTER_JSON)
    apps = []
    if data is None:
        print("[A] Master Spark tidak dapat diakses. Cek: Master sudah start? IP benar? Port 8080/firewall?")
    else:
        workers = [w for w in data.get("workers", []) if w.get("state") == "ALIVE"]
        total = sum(w.get("cores", 0) for w in workers)
        used = sum(w.get("coresused", 0) for w in workers)
        apps = sorted(data.get("activeapps", []), key=lambda a: a.get("name", ""))

        print(f"[A] CLUSTER MANAGER (Master) : {data.get('status')}  {data.get('url', '')}")
        print(f"    Worker ALIVE             : {len(workers)} / 4" + ("" if len(workers) >= 4 else "  <- ada worker belum join!"))
        print(f"    Core terpakai / total    : {used} / {total} [{bar(used, total, 20)}]")
        print(f"    Aplikasi aktif           : {len(apps)}")

        tampil = [w for w in workers if SHOW_IDLE_WORKERS or w.get("coresused", 0) > 0]
        tampil.sort(key=lambda x: x.get("host", ""))
        print(f"\n    {'WORKER':<24}{'CORES':<8}{'LOAD':<13}MEMORY(MB)")
        for w in tampil[:MAX_BARIS]:
            c, cu = w.get("cores", 0), w.get("coresused", 0)
            print(f"    {w.get('host', '?')[:23]:<24}{f'{cu}/{c}':<8}{bar(cu, c):<13}"
                  f"{w.get('memoryused', 0)}/{w.get('memory', 0)}")
        if len(tampil) > MAX_BARIS:
            print(f"    ... +{len(tampil) - MAX_BARIS} worker lain")
        if not tampil:
            print("    (tidak ada worker yang sedang sibuk)")

    # ---------------- [B] APLIKASI = DRIVER MAHASISWA ----------------
    print("\n[B] APLIKASI (1 mahasiswa = 1 Driver = 1 Application)")
    print(f"    {'NAMA APLIKASI':<30}{'STATE':<10}{'CORES':<7}{'EXEC':<6}{'JOB':<6}TASK selesai/total")
    infos = list(pool.map(baca_aplikasi, apps)) if apps else []
    detail = None
    for app, info in zip(apps, infos):
        nama = app.get("name", "?")
        awal = f"    {nama[:29]:<30}{app.get('state', '?'):<10}{app.get('cores', 0):<7}"
        if info is None:
            print(awal + f"{'-':<6}{'-':<6}(Spark UI 4040 tidak terjangkau)")
        else:
            done = sum(j.get("numCompletedTasks", 0) for j in info["jobs"])
            total = sum(j.get("numTasks", 0) for j in info["jobs"])
            print(awal + f"{len(info['execs']):<6}{len(info['jobs']):<6}{done}/{total}")
        if FOCUS and FOCUS.lower() in nama.lower():
            detail = (app, info)
    if not apps:
        print("    (belum ada aplikasi aktif)")

    # ---------------- [C] DETAIL SATU APLIKASI: JOB > STAGE > EXECUTOR ----------------
    if FOCUS:
        print("\n" + "-" * W)
        if detail is None:
            print(f"[C] DETAIL: tidak ada aplikasi aktif yang namanya memuat '{FOCUS}'")
        elif detail[1] is None:
            print(f"[C] DETAIL {detail[0].get('name')}: Spark UI Driver tidak terjangkau (firewall 4040?)")
        else:
            app, info = detail
            print(f"[C] DETAIL: {app.get('name')} ({app.get('id')})")

            print(f"    {'JOB':<5}{'STATUS':<11}{'STAGES':<12}{'TASK':<10}{'AKTIF':<7}LABEL")
            for j in sorted(info["jobs"], key=lambda x: x.get("jobId", 0))[-MAX_BARIS:]:
                task = f"{j.get('numCompletedTasks', 0)}/{j.get('numTasks', 0)}"
                label = j.get("description") or j.get("name", "")
                print(f"    {j.get('jobId'):<5}{j.get('status', ''):<11}{str(j.get('stageIds', '')):<12}"
                      f"{task:<10}{j.get('numActiveTasks', 0):<7}{str(label)[:40]}")

            print(f"\n    {'STAGE':<7}{'STATUS':<10}{'TASK':<10}{'SHUF.READ':<11}{'SHUF.WRITE':<12}NAMA")
            stages = sorted(info["stages"], key=lambda x: (x.get("stageId", 0), x.get("attemptId", 0)))
            for s in stages[-MAX_BARIS:]:
                task = f"{s.get('numCompleteTasks', 0)}/{s.get('numTasks', 0)}"
                print(f"    {s.get('stageId'):<7}{s.get('status', ''):<10}{task:<10}"
                      f"{ukuran(s.get('shuffleReadBytes')):<11}{ukuran(s.get('shuffleWriteBytes')):<12}"
                      f"{str(s.get('name', ''))[:35]}")

            print(f"\n    {'EXECUTOR (di worker mana)':<34}{'AKTIF':<7}{'SELESAI':<9}TOTAL")
            for e in info["execs"]:
                print(f"    {e.get('hostPort', '?'):<34}{e.get('activeTasks', 0):<7}"
                      f"{e.get('completedTasks', 0):<9}{e.get('totalTasks', 0)}")

    # ---------------- [D] LAPORAN TASK DARI EXECUTOR ----------------
    print("\n" + "-" * W)
    print("[D] LAPORAN TASK DARI EXECUTOR" + (f"  (filter Driver: '{FOCUS}')" if FOCUS else ""))
    with lock:
        snapshot = list(laporan.values())
    if FOCUS:
        snapshot = [t for t in snapshot if FOCUS.lower() in str(t.get("driver", "")).lower()]
    if not snapshot:
        print("    Belum ada laporan task.")
        return

    per_op = defaultdict(lambda: [0, 0, set()])
    per_worker = defaultdict(lambda: [0, 0])
    for t in snapshot:
        status = str(t.get("status", "")).upper()
        if status == "RUNNING":
            i = 0
        elif status == "DONE":
            i = 1
        else:
            continue
        op = per_op[t["app"]]
        op[i] += 1
        op[2].add(t.get("driver", "?"))
        per_worker[t["worker"]][i] += 1

    print(f"    {'OPERASI / LAB':<40}{'RUNNING':<9}{'DONE':<8}DRIVER")
    for op, (running, done, drv) in sorted(per_op.items())[:MAX_BARIS * 2]:
        pemilik = next(iter(drv)) if len(drv) == 1 else f"{len(drv)} mahasiswa"
        print(f"    {op[:39]:<40}{running:<9}{done:<8}{pemilik}")

    print(f"\n    {'EXECUTOR DI WORKER':<30}RUNNING / DONE")
    for w, (running, done) in sorted(per_worker.items(), key=lambda kv: -sum(kv[1]))[:MAX_BARIS]:
        print(f"    {str(w)[:29]:<30}{running:>7} / {done}")


try:
    while True:
        clear_output(wait=True)   # ANSI \033[2J / cls tidak berfungsi di Jupyter
        tampilkan()
        print(f"\nRefresh {REFRESH} detik | Interrupt (■) untuk berhenti | FOCUS = '{FOCUS}'")
        time.sleep(REFRESH)
except KeyboardInterrupt:
    print("\nMonitoring dihentikan (server laporan tetap berjalan).")

 MONITOR SPARK CLUSTER KELOMPOK (1 Master + 4 Worker)


In [ ]:
# ===== SEL 3 (opsional): lihat Driver terdaftar, reset laporan, matikan server =====
with lock:
    for app_id, d in sorted(drivers.items()):
        print(f"{app_id:<28}{d['host']:<22}{d['ui']}")
    laporan.clear()
print("Laporan task dikosongkan.")

# with lock: drivers.clear()                  # buka komentar untuk hapus daftar Driver
# server.shutdown(); server.server_close()    # buka komentar untuk mematikan server

### Tips mengajar
### Tips kelompok
- Isi `FOCUS = "PC-07"` (sebagian hostname anggota) lalu jalankan ulang Sel 2: panel **[C]** menampilkan Job, Stage, dan Executor anggota itu. Cocok untuk **Lab 3** (stage SKIPPED dan shuffle terlihat).
- **Pembagian core otomatis:** notebook Cluster memakai `total core worker / 5`. Contoh 16 core → 3 core/orang.
- Kolom EXEC/JOB berisi `-` → port 4040 PC Driver itu diblokir firewall.
- Aplikasi **WAITING** → core kelompok habis. Minta anggota lain menjalankan sel `spark.stop()`.
- Worker ALIVE < 4 → cek terminal Worker yang belum jalan / salah IP Master.